In [1]:
import torch
import torch.nn.functional as F
from data_generation.BetaVAE import *
from folktables import ACSDataSource, ACSIncome
import pandas as pd
from loaders.vae_loader import *

def load_acs_ca_year(year: int):
    data_source = ACSDataSource(survey_year=str(year), horizon="1-Year", survey="person")
    acs_data = data_source.get_data(states=["CA"], download=True)

 
    features = ACSIncome.features
    X, y, _ = ACSIncome.df_to_numpy(acs_data)

    X = pd.DataFrame(X, columns=features)
    y = pd.Series(y.astype(int), name="income")   
    return X, y

def make_train_test_split_like_paper():
    X14, y14 = load_acs_ca_year(2014)
    X18, y18 = load_acs_ca_year(2018)

    majority_mask_18 = (y18 == 0)

    X_train = pd.concat([X14, X18[majority_mask_18]], axis=0).reset_index(drop=True)
    y_train = pd.concat([y14, y18[majority_mask_18]], axis=0).reset_index(drop=True)

    X_test = X18.reset_index(drop=True)
    y_test = y18.reset_index(drop=True)
    return X_train, y_train, X_test, y_test



import torch

def kl_gaussian_class_cond(mu_flat, logvar_flat, mu_prior_batch):
    """
    mu_flat, logvar_flat : (B, D)
    mu_prior_batch       : (B, D)  mu_prior[y]
    KL(q || p) with p = N(mu_prior, I)
    Eq(5) in the paper (up to mean/sum conventions).
    """
    
    kl_per_sample = 0.5 * torch.sum(
        torch.exp(logvar_flat) + (mu_flat - mu_prior_batch) ** 2 - 1.0 - logvar_flat,
        dim=1
    )
    return kl_per_sample.mean()

def train_vae(
    vae,
    dataloader,
    mu_prior,     
    optimizer,
    device="cuda",
    beta=1.0,
):
    vae.train()
    running = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    n = 0

    mu_prior = mu_prior.to(device)  

    for x_num, x_cat, y in dataloader:
        x_num = x_num.to(device)
        x_cat = x_cat.to(device)
        y = y.to(device).long()

       
        x_num_hat, x_cat_logits, mu, logvar = vae(x_num, x_cat)

        
        B, M, d = mu.shape
        mu_flat = mu.view(B, M * d)
        logvar_flat = logvar.view(B, M * d)

        
        recon_num = F.mse_loss(x_num_hat, x_num, reduction="mean")

        rec_cat = 0.0
        for j, logits in enumerate(x_cat_logits):
            rec_cat += F.cross_entropy(logits, x_cat[:, j], reduction="mean")

        if len(x_cat_logits) > 0:
            rec_cat /= len(x_cat_logits)

        loss_recon = recon_num + rec_cat

       
        mu_prior_batch = mu_prior[y]  
        loss_kl = kl_gaussian_class_cond(mu_flat, logvar_flat, mu_prior_batch)

        loss = loss_recon + beta * loss_kl

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        B = x_num.size(0)
        running["loss"] += loss.item() * B
        running["recon"] += loss_recon.item() * B
        running["kl"] += loss_kl.item() * B
        n += B

    for k in running:
        running[k] /= max(1, n)
    return running

In [2]:
X_train, y_train, X_test, y_test = make_train_test_split_like_paper()  # CA 2014 + maj 2018 / test = 2018 :contentReference[oaicite:1]{index=1}


cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]


df_train = X_train.copy()
df_train["y"] = y_train.values

df_test = X_test.copy()
df_test["y"] = y_test.values


loaders_train, prepro, meta = make_tabular_loaders(
    df=df_train,
    num_cols=num_cols,
    cat_cols=cat_cols,
    label_col="y",
    batch_size=256,
    test_size=None,  
    val_size=0.1,
)

train_loader = loaders_train["train"]
val_loader   = loaders_train["val"]


loaders_test, _, _ = make_tabular_loaders(
    df=df_test,
    num_cols=num_cols,
    cat_cols=cat_cols,
    label_col="y",
    batch_size=256,
    test_size=None,
    val_size=0.0,
    num_scaler=prepro["num_scaler"],
    cat_encoder=prepro["cat_encoder"],
)

test_loader = loaders_test["train"]  

In [3]:
num_numerical = len(num_cols)
cat_cardinalities = meta["cat_cardinalities"]

d = 32  # embedding dim (à choisir)
beta = 1.0
device = "cuda" if torch.cuda.is_available() else "cpu"



vae = BetaVAE(num_numerical=num_numerical, cat_cardinalities=cat_cardinalities, d=d, beta=beta).to(device)
optimizer = torch.optim.AdamW(vae.parameters(), lr=1e-3, weight_decay=1e-4)

/Users/feddy/anaconda3/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [4]:
@torch.no_grad()
def estimate_mu_prior(vae, loader, device):
    vae.eval()

    D = vae.M * vae.d
    sum0 = torch.zeros(D, device=device)
    sum1 = torch.zeros(D, device=device)
    n0 = 0
    n1 = 0

    for x_num, x_cat, y in loader:
        x_num = x_num.to(device)
        x_cat = x_cat.to(device)
        y = y.to(device).long()

        _, _, mu, _ = vae(x_num, x_cat) 

        B, M, d = mu.shape
        mu_flat = mu.view(B, M * d)

        m0 = (y == 0)
        m1 = (y == 1)

        if m0.any():
            sum0 += mu_flat[m0].sum(dim=0)
            n0 += int(m0.sum())

        if m1.any():
            sum1 += mu_flat[m1].sum(dim=0)
            n1 += int(m1.sum())

    mu0 = sum0 / max(1, n0)
    mu1 = sum1 / max(1, n1)

    vae.train()
    return torch.stack([mu0, mu1], dim=0)  

In [5]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def eval_vae(vae, dataloader, mu_prior, device="cuda", beta=1.0):
    vae.eval()
    mu_prior = mu_prior.to(device)

    running = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    n = 0

    for x_num, x_cat, y in dataloader:
        x_num = x_num.to(device)
        x_cat = x_cat.to(device)
        y = y.to(device).long()

       
        x_num_hat, x_cat_logits, mu, logvar = vae(x_num, x_cat)

    
        recon_num = F.mse_loss(x_num_hat, x_num, reduction="mean")

        rec_cat = 0.0
        for j, logits in enumerate(x_cat_logits):
            rec_cat += F.cross_entropy(logits, x_cat[:, j], reduction="mean")

        if len(x_cat_logits) > 0:
            rec_cat /= len(x_cat_logits)

        loss_recon = recon_num + rec_cat


        B, M, d = mu.shape
        mu_flat = mu.view(B, M * d)
        logvar_flat = logvar.view(B, M * d)

        mu_prior_y = mu_prior[y]  

        loss_kl = 0.5 * torch.mean(
            torch.sum(
                torch.exp(logvar_flat)
                + (mu_flat - mu_prior_y) ** 2
                - 1.0
                - logvar_flat,
                dim=1
            )
        )

        loss = loss_recon + beta * loss_kl

        Bsz = x_num.size(0)
        running["loss"] += loss.item() * Bsz
        running["recon"] += loss_recon.item() * Bsz
        running["kl"] += loss_kl.item() * Bsz
        n += Bsz

    for k in running:
        running[k] /= max(1, n)

    vae.train()
    return running

In [ ]:
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
best_val = float("inf")
SAVE_DIR = Path("checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

for epoch in range(1, 51):
    mu_prior = estimate_mu_prior(vae, train_loader, device)

    train_stats = train_vae(
        vae, train_loader, mu_prior, optimizer, device=device, beta=vae.beta
    )

    if val_loader is not None:
        val_stats = eval_vae(vae, val_loader, mu_prior, device)
        val_loss = val_stats["loss"]
    else:
        val_loss = train_stats["loss"]

    if val_loss < best_val:
        best_val = val_loss
        torch.save(
            {
                "vae_state": vae.state_dict(),
                "mu_prior": mu_prior.detach().cpu(),
            },
            SAVE_DIR / "best_vae.pt"
        )

    print(f"Epoch {epoch} | train={train_stats['loss']:.4f} val={val_loss:.4f}")

/Users/feddy/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1 | train=4.6728 val=2.0212
